In [2]:
"""
Step 1: Create the DuckDB database
-----------------------------------
Run this once to set up nmfp.duckdb with the exact schema
from the real SEC N-MFP TSV files.

Join key:         ACCESSION_NUMBER (present in both files)
Fund type filter: MONEYMARKETFUNDCATEGORY = 'Prime'

Usage:
    pip install duckdb
    python create_db.py
"""

import duckdb
import os

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")


def create_db():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Deleted existing {DB_PATH}\n")

    print(f"Creating {DB_PATH} ...\n")
    con = duckdb.connect(DB_PATH)

    # ── Table 1: series_info ─────────────────────────────────────────────
    # From NMFP_SERIESLEVELINFO.tsv
    # One row per fund series per monthly ZIP.
    # MONEYMARKETFUNDCATEGORY values: 'Prime', 'Government', 'Other Tax Exempt', 'Single State'
    con.execute("""
        CREATE TABLE series_info (
            -- Identity
            ACCESSION_NUMBER                VARCHAR,
            SECURITIESACTFILENUMBER         VARCHAR,

            -- Fund metadata
            INDPPUBACCTNAME                 VARCHAR,   -- independent public accountant
            INDPPUBACCTCITY                 VARCHAR,
            INDPPUBACCTSTATECOUNTRY         VARCHAR,
            FEEDERFUNDFLAG                  VARCHAR,
            MASTERFUNDFLAG                  VARCHAR,
            SERIESFUNDINSUCMPNYSEPACCNTFLA  VARCHAR,
            MONEYMARKETFUNDCATEGORY         VARCHAR,   -- 'Prime' | 'Government' | 'Other Tax Exempt' | 'Single State'
            FUNDEXEMPTRETAILFLAG            VARCHAR,
            FUNDRETAILMONEYMARKETFLAG       VARCHAR,
            GOVMONEYMRKTFUNDFLAG            VARCHAR,

            -- Portfolio maturity
            AVERAGEPORTFOLIOMATURITY        DOUBLE,
            AVERAGELIFEMATURITY             DOUBLE,

            -- Weekly daily liquid assets ($ total, 5 Fridays per month)
            TOTDLYLIQUIDASSETFRIDAYWEEK1    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK2    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK3    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK4    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK5    DOUBLE,

            -- Weekly weekly liquid assets ($ total, 5 Fridays)
            TOTWLYLIQUIDASSETFRIDAYWEEK1    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK2    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK3    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK4    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK5    DOUBLE,

            -- Daily liquid asset % (5 Fridays)
            PCTDLYLIQUIDASSETFRIDAYWEEK1    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK2    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK3    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK4    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK5    DOUBLE,

            -- Weekly liquid asset % (5 Fridays)
            PCTWKLYLIQUIDASSETFRIDAYWEEK1   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK2   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK3   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK4   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK5   DOUBLE,

            -- Portfolio values
            CASH                            DOUBLE,
            TOTALVALUEPORTFOLIOSECURITIES   DOUBLE,
            AMORTIZEDCOSTPORTFOLIOSECURITI  DOUBLE,
            TOTALVALUEOTHERASSETS           DOUBLE,
            TOTALVALUELIABILITIES           DOUBLE,
            NETASSETOFSERIES                DOUBLE,
            NUMBEROFSHARESOUTSTANDING       DOUBLE,

            -- Pricing
            SEEKSTABLEPRICEPERSHARE         VARCHAR,
            STABLEPRICEPERSHARE             DOUBLE,
            SEVENDAYGROSSYIELD              DOUBLE,

            -- NAV per share (5 Fridays)
            NETASSETVALUEFRIDAYWEEK1        DOUBLE,
            NETASSETVALUEFRIDAYWEEK2        DOUBLE,
            NETASSETVALUEFRIDAYWEEK3        DOUBLE,
            NETASSETVALUEFRIDAYWEEK4        DOUBLE,
            NETASSETVALUEFRIDAYWEEK5        DOUBLE,

            -- Flags
            CASHMGMTVEHICLEAFFLIATEDFUNDF   VARCHAR,
            LIQUIDITYFEEFUNDAPPLYFLAG       VARCHAR,

            -- ETL metadata
            SOURCE_PERIOD                   VARCHAR    -- e.g. '2026-04-09 to 2026-05-07'
        )
    """)
    print("Created table: series_info  (51 columns + SOURCE_PERIOD)")

    # ── Table 2: liquidity ───────────────────────────────────────────────
    # From NMFP_LIQUIDASSETSDETAILS.tsv
    # ~20 rows per accession number (one per calendar day in the reporting month)
    con.execute("""
        CREATE TABLE liquidity (
            ACCESSION_NUMBER            VARCHAR,
            TOTVALUEDAILYLIQUIDASSETS   DOUBLE,    -- $ total daily liquid assets
            TOTVALUEWEEKLYLIQUIDASSETS  DOUBLE,    -- $ total weekly liquid assets
            PCTDAILYLIQUIDASSETS        DOUBLE,    -- % of net assets (e.g. 0.9784 = 97.84%)
            PCTWEEKLYLIQUIDASSETS       DOUBLE,    -- % of net assets
            TOTLIQUIDASSETSNEARPCTDATE  VARCHAR,   -- date: 'DD-MON-YYYY' e.g. '15-MAY-2026'

            -- ETL metadata
            SOURCE_PERIOD               VARCHAR
        )
    """)
    print("Created table: liquidity  (6 columns + SOURCE_PERIOD)")

    # ── Indexes ──────────────────────────────────────────────────────────
    con.execute("CREATE INDEX idx_series_accession  ON series_info (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_series_category   ON series_info (MONEYMARKETFUNDCATEGORY)")
    con.execute("CREATE INDEX idx_series_period     ON series_info (SOURCE_PERIOD)")
    con.execute("CREATE INDEX idx_liq_accession     ON liquidity   (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_liq_period        ON liquidity   (SOURCE_PERIOD)")
    con.execute("CREATE INDEX idx_liq_date ON liquidity (TOTLIQUIDASSETSNEARPCTDATE)")
    print("Created indexes")

    # ── Load tracking table ──────────────────────────────────────────────
    con.execute("""
        CREATE TABLE loaded_periods (
            SOURCE_PERIOD   VARCHAR PRIMARY KEY,
            ZIP_URL         VARCHAR,
            LOADED_AT       TIMESTAMP DEFAULT current_timestamp,
            SERIES_ROWS     INTEGER,
            LIQUIDITY_ROWS  INTEGER
        )
    """)
    print("Created table: loaded_periods\n")

    # ── Verify ───────────────────────────────────────────────────────────
    tables = con.execute("SHOW TABLES").fetchall()
    print("Tables in database:")
    for (t,) in tables:
        cols = con.execute(f"DESCRIBE {t}").fetchall()
        print(f"  {t}  ({len(cols)} columns)")

    con.close()
    size = os.path.getsize(DB_PATH)
    print(f"\nDatabase ready: {os.path.abspath(DB_PATH)}  ({size:,} bytes)")
    print("\nNext step: run etl.py to populate it.")


if __name__ == "__main__":
    create_db()

Creating /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/nmfp.duckdb ...

Created table: series_info  (51 columns + SOURCE_PERIOD)
Created table: liquidity  (6 columns + SOURCE_PERIOD)
Created indexes
Created table: loaded_periods

Tables in database:
  liquidity  (7 columns)
  loaded_periods  (5 columns)
  series_info  (52 columns)

Database ready: /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/nmfp.duckdb  (274,432 bytes)

Next step: run etl.py to populate it.


In [3]:
import duckdb
import os

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")

con = duckdb.connect(DB_PATH)

# Show all tables
print("Tables:")
print(con.execute("SHOW TABLES").df())

# Show columns for each table
for table in ["series_info", "liquidity", "loaded_periods"]:
    print(f"\n{table} columns:")
    print(con.execute(f"DESCRIBE {table}").df())

con.close()

Tables:
             name
0       liquidity
1  loaded_periods
2     series_info

series_info columns:
                       column_name column_type null   key default extra
0                 ACCESSION_NUMBER     VARCHAR  YES  None    None  None
1          SECURITIESACTFILENUMBER     VARCHAR  YES  None    None  None
2                  INDPPUBACCTNAME     VARCHAR  YES  None    None  None
3                  INDPPUBACCTCITY     VARCHAR  YES  None    None  None
4          INDPPUBACCTSTATECOUNTRY     VARCHAR  YES  None    None  None
5                   FEEDERFUNDFLAG     VARCHAR  YES  None    None  None
6                   MASTERFUNDFLAG     VARCHAR  YES  None    None  None
7   SERIESFUNDINSUCMPNYSEPACCNTFLA     VARCHAR  YES  None    None  None
8          MONEYMARKETFUNDCATEGORY     VARCHAR  YES  None    None  None
9             FUNDEXEMPTRETAILFLAG     VARCHAR  YES  None    None  None
10       FUNDRETAILMONEYMARKETFLAG     VARCHAR  YES  None    None  None
11            GOVMONEYMRKTFUNDFLAG

In [4]:
"""
Step 2: ETL — Load a single ZIP into nmfp.duckdb
--------------------------------------------------
Reads NMFP_SERIESLEVELINFO.tsv and NMFP_LIQUIDASSETSDETAILS.tsv
from a local ZIP, filters for Prime funds, and inserts into the DB.

Usage:
    pip install duckdb pandas
    Run all cells in order.
"""
import os
import duckdb
import pandas as pd

# ── Config ───────────────────────────────────────────────────────────────────
DB_PATH     = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
FOLDER_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/20260508-20260605_nmfp")
SOURCE_PERIOD = "20260508-20260605"

SERIES_FILE    = "NMFP_SERIESLEVELINFO.tsv"
LIQUIDITY_FILE = "NMFP_LIQUIDASSETSDETAILS.tsv"
# ─────────────────────────────────────────────────────────────────────────────


def align_to_schema(con, table, df):
    schema_cols = [row[0].upper() for row in con.execute(f"DESCRIBE {table}").fetchall()]
    for col in schema_cols:
        if col not in df.columns:
            df[col] = None
    return df[[c for c in schema_cols if c in df.columns]]


def load_folder(folder_path, db_path, source_period):
    print(f"Folder:   {folder_path}")
    print(f"Database: {db_path}")
    print(f"Period:   {source_period}\n")

    con = duckdb.connect(db_path)

    # Check if already loaded
    already = con.execute(
        "SELECT COUNT(*) FROM loaded_periods WHERE SOURCE_PERIOD = ?", [source_period]
    ).fetchone()[0]
    if already:
        print(f"[skip] '{source_period}' already in database.")
        con.close()
        return

    # ── 1. Series info → filter Prime ────────────────────────────────────
    print("Reading NMFP_SERIESLEVELINFO.tsv ...")
    series_df = pd.read_csv(os.path.join(folder_path, SERIES_FILE), sep="\t", low_memory=False, on_bad_lines="warn")
    series_df.columns = series_df.columns.str.strip().str.upper()
    print(f"  Total rows: {len(series_df):,}")
    print(f"  Fund categories:\n{series_df['MONEYMARKETFUNDCATEGORY'].value_counts().to_string()}\n")

    prime_df = series_df[series_df["MONEYMARKETFUNDCATEGORY"] == "Prime"].copy()
    print(f"  Prime funds: {len(prime_df):,}")

    if prime_df.empty:
        print("[error] No Prime funds found.")
        con.close()
        return

    prime_accessions = set(prime_df["ACCESSION_NUMBER"].dropna().unique())
    prime_df["SOURCE_PERIOD"] = source_period

    # ── 2. Liquidity → filter to Prime accession numbers ─────────────────
    print("\nReading NMFP_LIQUIDASSETSDETAILS.tsv ...")
    liq_df = pd.read_csv(os.path.join(folder_path, LIQUIDITY_FILE), sep="\t", low_memory=False, on_bad_lines="warn")
    liq_df.columns = liq_df.columns.str.strip().str.upper()
    print(f"  Total rows: {len(liq_df):,}")

    prime_liq = liq_df[liq_df["ACCESSION_NUMBER"].isin(prime_accessions)].copy()
    prime_liq["SOURCE_PERIOD"] = source_period
    print(f"  Prime liquidity rows: {len(prime_liq):,}")

    # ── 3. Align to schema and insert ────────────────────────────────────
    print("\nInserting into database ...")
    prime_df  = align_to_schema(con, "series_info", prime_df)
    prime_liq = align_to_schema(con, "liquidity",   prime_liq)

    con.execute("INSERT INTO series_info SELECT * FROM prime_df")
    con.execute("INSERT INTO liquidity   SELECT * FROM prime_liq")

    # ── 4. Record in loaded_periods ──────────────────────────────────────
    con.execute(
        "INSERT INTO loaded_periods (SOURCE_PERIOD, ZIP_URL, SERIES_ROWS, LIQUIDITY_ROWS) VALUES (?, ?, ?, ?)",
        [source_period, folder_path, len(prime_df), len(prime_liq)],
    )

    con.close()
    print(f"\n✓ Done.")
    print(f"  series_info rows inserted : {len(prime_df):,}")
    print(f"  liquidity rows inserted   : {len(prime_liq):,}")


load_folder(FOLDER_PATH, DB_PATH, SOURCE_PERIOD)

Folder:   /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/20260508-20260605_nmfp
Database: /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/nmfp.duckdb
Period:   20260508-20260605

Reading NMFP_SERIESLEVELINFO.tsv ...
  Total rows: 323
  Fund categories:
MONEYMARKETFUNDCATEGORY
Government          240
Prime                42
Other Tax Exempt     25
Single State         16

  Prime funds: 42

Reading NMFP_LIQUIDASSETSDETAILS.tsv ...
  Total rows: 6,449
  Prime liquidity rows: 840

Inserting into database ...

✓ Done.
  series_info rows inserted : 42
  liquidity rows inserted   : 840


In [7]:
import duckdb, os

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
con = duckdb.connect(DB_PATH)

# Row counts
print("series_info rows:", con.execute("SELECT COUNT(*) FROM series_info").fetchone()[0])
print("liquidity rows:  ", con.execute("SELECT COUNT(*) FROM liquidity").fetchone()[0])

# Confirm only Prime funds loaded
print("\nFund categories:")
print(con.execute("SELECT MONEYMARKETFUNDCATEGORY, COUNT(*) as n FROM series_info GROUP BY 1").df())

# Confirm period recorded
print("\nLoaded periods:")
print(con.execute("SELECT * FROM loaded_periods").df())

# Check for NULLs in key columns
print("\nNull counts in liquidity:")
print(con.execute("""
    SELECT 
        COUNT(*) - COUNT(ACCESSION_NUMBER)        AS null_accession,
        COUNT(*) - COUNT(TOTLIQUIDASSETSNEARPCTDATE) AS null_date,
        COUNT(*) - COUNT(PCTDAILYLIQUIDASSETS)    AS null_pct_daily,
        COUNT(*) - COUNT(PCTWEEKLYLIQUIDASSETS)   AS null_pct_weekly
    FROM liquidity
""").df())

# Sample joined data
print("\nSample joined data:")
print(con.execute("""
    SELECT s.ACCESSION_NUMBER, s.INDPPUBACCTNAME, l.TOTLIQUIDASSETSNEARPCTDATE,
           l.PCTDAILYLIQUIDASSETS, l.PCTWEEKLYLIQUIDASSETS
    FROM liquidity l
    JOIN series_info s ON l.ACCESSION_NUMBER = s.ACCESSION_NUMBER
    LIMIT 5
""").df())

con.close()

series_info rows: 42
liquidity rows:   840

Fund categories:
  MONEYMARKETFUNDCATEGORY   n
0                   Prime  42

Loaded periods:
       SOURCE_PERIOD                                            ZIP_URL  \
0  20260508-20260605  /Users/saahilkajarekar/Downloads/HackYourSumme...   

                   LOADED_AT  SERIES_ROWS  LIQUIDITY_ROWS  
0 2026-07-01 02:30:14.898117           42             840  

Null counts in liquidity:
   null_accession  null_date  null_pct_daily  null_pct_weekly
0               0          0               0                0

Sample joined data:
       ACCESSION_NUMBER     INDPPUBACCTNAME TOTLIQUIDASSETSNEARPCTDATE  \
0  0001410368-26-057221  Ernst & Young  LLP                01-MAY-2026   
1  0001410368-26-057221  Ernst & Young  LLP                04-MAY-2026   
2  0001410368-26-057221  Ernst & Young  LLP                05-MAY-2026   
3  0001410368-26-057221  Ernst & Young  LLP                06-MAY-2026   
4  0001410368-26-057221  Ernst & Young  LLP      